In [1]:
%%capture
%pip install black[jupyter] blackcellmagic

In [2]:
import IPython
import black

# from google.colab import output


def format_cell(result):
    shell = IPython.get_ipython()
    raw_cell = shell.history_manager.input_hist_raw[-1]

    try:
        formatted = black.format_str(raw_cell, mode=black.Mode())
        if formatted.strip() != raw_cell.strip():
            # We update the cell content for the next time it is viewed/edited
            shell.set_next_input(formatted, replace=True)
    except Exception as e:
        # Skip formatting if there's a syntax error during the black process
        pass


# Register the hook to run after every cell execution
get_ipython().events.register("post_run_cell", format_cell)
print("Auto-formatter active: Cells will reformat automatically upon execution.")

Auto-formatter active: Cells will reformat automatically upon execution.


In [3]:
%%capture
%pip install uaibot

In [5]:
import numpy as np
import uaibot as ub
import qpsolvers

from typing import cast
from uaibot.simobjects import SimObject

robot = ub.Robot.create_franka_emika_3()

# Simulation parameters
dt = 0.01
tmax = 12
n = np.shape(robot.q)[0]
m = 4  # Task space dimension

# --- State Management ---
q_current = np.copy(robot.q)

# --- Joint Limit Parameters ---
q_limits = np.array(robot._joint_limit)
q_min = q_limits[:, 0].reshape((n, 1))
q_max = q_limits[:, 1].reshape((n, 1))
q_mean = (q_max + q_min) / 2.0
q_range_sq = np.square(q_max - q_min)

# --- Path Definition ---
R = 0.1
x_c = 0.2
y_c = 0.0
z_c = 1.0
omega_d = np.pi / 2

s_d = lambda p: np.matrix(
    [x_c, y_c + R * np.cos(p), z_c + R * np.sin(p)]
).reshape((3, 1))

s_d_prime = lambda p: np.matrix(
    [0, -R * np.sin(p), R * np.cos(p)]
).reshape((3, 1))

z_d = np.matrix([1, 0, 0]).reshape((3, 1))

pc = np.matrix(np.zeros((3, 0)))
for i in range(200):
    pc = np.block([pc, s_d(2 * np.pi * i / 199)])

target_pos_pc = ub.PointCloud(points=pc, size=0.03, color="purple")
ball_tr = ub.Ball(htm=np.asmatrix(np.identity(4)), radius=0.02, color="cyan")
sim = ub.Simulation(cast(list[SimObject], [robot, target_pos_pc, ball_tr]))

def fun_F(r):
    K = 0.25  # Gain for the nonlinear feedback
    r_norm = np.linalg.norm(r)
    return - K * np.sqrt(r_norm) * np.sign(r)

# --- Control Loop ---
k0 = 1.0         
gamma = 500.0    
epsilon = 1e-3   
lambda_d = 0.01  # Uniform damping scalar (replaces Q and R matrices)

p = 0.0          
t = 0.0

# Pre-allocate static QP Objective Matrix
P_qp = np.block([
    [lambda_d * np.identity(n), np.zeros((n, m))],
    [np.zeros((m, n)),          np.identity(m)]
])

for _ in range(round(tmax / dt)):
    Jg, fk = robot.jac_geo(q_current)
    z_e = fk[0:3, 2]
    s_e = fk[0:3, 3]

    r = np.matrix(np.zeros((4, 1)))
    r[0:3] = s_e - s_d(p)
    r[3] = 1 - z_d.T * z_e
    
    r_norm_sq = (r.T * r)[0, 0]
    p_dot = omega_d / (1.0 + gamma * r_norm_sq)

    Jr = np.matrix(np.zeros((4, n)))
    ff = np.vstack((-s_d_prime(p) * p_dot, np.zeros((1, 1))))

    Jr[0:3, :] = Jg[0:3, :]
    Jr[3, :] = z_d.T * ub.Utils.S(z_e) * Jg[3:6, :]

    # Secondary joint limit avoidance objective
    grad_H = np.matrix((q_current - q_mean) / q_range_sq)
    q_dot_0 = -k0 * grad_H

    # --- Augmented QP Formulation ---
    q_vec_qp = np.concatenate([-lambda_d * np.asarray(q_dot_0).flatten(), np.zeros(m)])
    
    # Equality constraints: Jr*q_dot - I*w = b
    A_qp = np.hstack([np.asarray(Jr), -np.identity(m)])
    b_qp = np.asarray(fun_F(r) - ff).flatten()
    
    # Inequality constraints (Slack variables 'w' are unbounded)
    lb_dq = np.asarray(q_min - q_current).flatten() / dt + epsilon
    ub_dq = np.asarray(q_max - q_current).flatten() / dt - epsilon
    
    lb_qp = np.concatenate([lb_dq, np.full(m, -np.inf)])
    ub_qp = np.concatenate([ub_dq, np.full(m, np.inf)])

    # Solve Unconditionally Feasible Optimization
    try:
        u_opt = qpsolvers.solve_qp(
            P=P_qp, 
            q=q_vec_qp, 
            A=A_qp, 
            b=b_qp, 
            lb=lb_qp, 
            ub=ub_qp, 
            solver="osqp"
        )
        
        if u_opt is None:
            # Mathematical fallback: this condition is analytically unreachable unless P is ill-conditioned
            print(f"Solver internal failure at t={t:.2f}. Halting.")
            break
            
        # Extract only the joint velocities; discard the utilized slack vector
        u = np.matrix(u_opt[:n]).reshape((n, 1))

    except Exception as e:
        print(f"Optimization Exception: {e}")
        break

    # State Integration
    q_current = q_current + u * dt
    p = p + p_dot * dt
    t = t + dt
    
    robot.add_ani_frame(time=t, q=q_current)
    ball_tr.add_ani_frame(time=t, htm=ub.Utils.trn(s_d(p)))

sim.run()